# ScGeo Phase 1 — GSE280305 QC (Raw vs Scanorama)

**Goal:** reproduce the PBMC QC-style comparison on GSE280305 using ScGeo.
- **Batch-mixing QC:** `mixscore` on **sample**
- **Biology contrast QC:** define **phase** (Early vs Late) and compute:
  - `shift` (Early→Late) **by leiden**
  - `density_overlap` (Early vs Late) **by leiden**

This notebook is intentionally light-weight: it avoids concatenating AnnData objects and only operates on existing `obsm` representations.


In [ ]:
import scanpy as sc
import numpy as np
import scgeo as sg

sc.set_figure_params(dpi=120, frameon=False)


In [ ]:
# Inputs (adjust if needed)
AB_IN = "data/GSE280305_paths_AB.h5ad"          # contains X_pca_raw, X_scanorama, X_umap_raw, X_umap_scanorama, leiden_raw
# Optionally also load scalars/h5ad with CellRank outputs elsewhere; Phase 1 QC does not require it.

adata = sc.read_h5ad(AB_IN)
adata


In [ ]:
# Quick sanity checks: required keys
need_obs = ["timepoint", "sample", "leiden_raw"]
need_obsm = ["X_umap_raw", "X_umap_scanorama", "X_pca_raw", "X_scanorama"]

missing_obs = [k for k in need_obs if k not in adata.obs]
missing_obsm = [k for k in need_obsm if k not in adata.obsm]

print("missing obs:", missing_obs)
print("missing obsm:", missing_obsm)


In [ ]:
# Define a binary biology contrast: Early vs Late
tp = adata.obs["timepoint"].astype(str)
phase = np.where(tp.isin(["D8", "D11"]), "Early", np.where(tp.isin(["D14", "D21"]), "Late", "Other"))
adata.obs["phase"] = phase
adata.obs["phase"] = adata.obs["phase"].astype("category")

adata.obs["phase"].value_counts()


In [ ]:
# Baseline UMAP views (Raw vs Scanorama)
sc.pl.embedding(adata, basis="X_umap_raw", color=["timepoint",  "leiden_raw"], wspace=0.45)
sc.pl.embedding(adata, basis="X_umap_scanorama", color=["timepoint",  "leiden_raw"], wspace=0.45)


## 1) Batch-mixing QC: mixscore on **sample**

This tests whether **technical batches (samples)** are locally mixed.
Run it on both embeddings (Raw vs Scanorama) and visualize on the corresponding UMAP.


In [ ]:
# mixscore expects label_key (NOT condition_key).
# IMPORTANT: set obs_key so the result is written to adata.obs[obs_key] for plotting.

# RAW
sg.tl.mixscore(
    adata,
    label_key="timepoint",
    rep="X_umap_raw",
    obs_key="mix_sample_raw",
    store_key="mix_sample_raw",
)

# SCANORAMA
sg.tl.mixscore(
    adata,
    label_key="timepoint",
    rep="X_umap_scanorama",
    obs_key="mix_sample_scanorama",
    store_key="mix_sample_scanorama",
)

adata.obs[["mix_sample_raw", "mix_sample_scanorama"]].describe()


In [ ]:
# Plot mixscore on the correct embedding
sg.pl.score_embedding(adata, score_key="mix_sample_raw", basis="umap_raw", title="ScGeo mixscore (sample) — Raw UMAP")
sg.pl.score_embedding(adata, score_key="mix_sample_scanorama", basis="umap_scanorama", title="ScGeo mixscore (sample) — Scanorama UMAP")


## 1.5) Where is batch mixing happening?

Use the mixscore maps to highlight regions where **sample labels** are locally well-mixed (good batch correction) vs separated (potential batch structure).

In [ ]:
# Density-style view (optional): where do different labels concentrate?
# NOTE: `embedding_density` doesn't take a `title=` kwarg.
sg.pl.embedding_density(adata, basis="umap_raw", groupby="timepoint")
sg.pl.embedding_density(adata, basis="umap_scanorama", groupby="timepoint")


## 2) Biology contrast QC: shift Early→Late **by leiden**

This quantifies the magnitude/direction of the Early→Late displacement, *within each leiden cluster*.
Run it on:
- `X_pca_raw` (baseline)
- `X_scanorama` (corrected space)


In [ ]:
# SHIFT (Early -> Late) by cluster in RAW PCA space
sg.tl.shift(
    adata,
    rep="X_pca_raw",
    condition_key="phase",
    group0="Early",
    group1="Late",
    by="leiden_raw",
    store_key="shift_phase_raw",
)

# SHIFT (Early -> Late) by cluster in Scanorama space
sg.tl.shift(
    adata,
    rep="X_scanorama",
    condition_key="phase",
    group0="Early",
    group1="Late",
    by="leiden_raw",
    store_key="shift_phase_scanorama",
)

print("keys:", adata.uns["shift_phase_raw"].keys(), adata.uns["shift_phase_scanorama"].keys())


In [ ]:
# Rank clusters by ||Δ|| (more informative than level='global' here)
ax1 = sg.pl.delta_rank(adata, store_key="shift_phase_raw", kind="shift", level="by", top_n=None)
ax1.set_title("Shift magnitude ranking (Early→Late) — RAW PCA (by leiden_raw)")

ax2 = sg.pl.delta_rank(adata, store_key="shift_phase_scanorama", kind="shift", level="by", top_n=None)
ax2.set_title("Shift magnitude ranking (Early→Late) — Scanorama space (by leiden_raw)")


In [ ]:
# Save a light Phase-1 QC artifact (keeps existing reps + new obs columns + uns outputs)
OUT = "data/GSE280305_scgeo_phase1_qc.h5ad"
adata.write(OUT)
print("wrote:", OUT)


In [ ]:
condition_key="phase"
group0="Early"
group1="Late"
cluster_key="leiden_raw"

In [ ]:
sg.tl.density_overlap(
    adata,
    rep='X_umap',
    condition_key=condition_key,
    group0=group0,
    group1=group1,
    by=cluster_key,
    store_key='scgeo',
)

In [ ]:
# Global + per-cluster summary bars
sg.pl.density_overlap(adata, store_key='density_overlap', level='global')
sg.pl.density_overlap(adata, store_key='density_overlap', level='by')


In [ ]:
sg.pl.embedding_density(
    adata,
    groupby=cluster_key,
    basis="umap_scanorama",
    contour=False,
    background="white",
    mask_zeros=True,      # <-- critical
    imshow_alpha=1.0,

)



In [ ]:
sg.pl.embedding_density(
    adata,
    groupby=cluster_key,
    basis="umap_scanorama",
    contour=True,
    smooth_k=9,
    contour_levels=8,
    contour_linewidth=0.5,
    mask_zeros=True,
    log1p=True,
    cmap='RdBu_r'
)



In [ ]:
sg.tl.distribution_test(
    adata,
    rep='X_pca',
    condition_key=condition_key,
    group0=group0,
    group1=group1,
    by=cluster_key,
    method='energy',
    n_perm=200,  # keep demo fast; increase for publication
    seed=0,
    store_key='scgeo',
)


In [ ]:
# Summary table plots
sg.pl.distribution_test(adata, store_key='distribution_test', level='global')
sg.pl.distribution_test(adata, store_key='distribution_test', level='by')


In [ ]:
sg.tl.consensus_subspace(
    adata,
    rep='X_pca',
    condition_key=condition_key,
    group0=group0,
    group1=group1,
    sample_key=None,   # set if you have donors/samples
    n_components=2,
    obs_key_prefix='cs',
    min_cells=20,
)


In [ ]:
sg.pl.consensus_subspace_panel(adata, score_key='cs_score', basis="umap_scanorama", topk=200)
sg.pl.highlight_topk_cells(adata, score_key='cs_score', basis="umap_scanorama", topk=200, title='Top consensus-driving cells')


In [ ]:
sg.pl.highlight_topk_cells(
    adata,
    score_key="cs_score",
    basis="umap_scanorama",
    topk=50,
    groupby="leiden_raw",
    outline_topk=True,
    hi_size=36,
    outline_lw=1.8,
)


In [ ]:
fig, ax, stats, legend = sg.pl.consensus_structure(
    adata,
    score_key="cs_score",
    basis="umap_scanorama",
    groupby="leiden_raw",
    topk=100,
    hi_size=36,
    outline_lw=1.8,
    print_legend=True,   # prints summary lines
)


sg.pl.legend_from_data(legend)

In [ ]:
sc.pp.neighbors(adata)

In [ ]:
sc.tl.paga(adata,groups="leiden_raw")

In [ ]:
sc.pl.paga(adata)

In [ ]:
sg.pl.paga_shift_map(
    adata,
    node_key="leiden_raw",
    condition_key="timepoint",
    group0="D8",
    group1="D21",
    basis="umap_scanorama",
)

In [ ]:
sg.pl.paga_shift_map(
    adata,
    node_key="leiden_raw",
    condition_key="timepoint",
    group0="D8",
    group1="D21",
    basis="umap_scanorama",
    connectivity_threshold=0.15,
    arrow_scale=1.4,
    arrow_width=0.01,
    label_top_n=8,
)

In [ ]:
sg.pl.composition_drift(
    adata,
    node_key="leiden_raw",
    condition_key="timepoint",
    group0="D8",
    group1="D21",
    basis="umap_scanorama",
)

In [ ]:
sg.pl.recovery_compass(
    adata,
    node_key="leiden_raw",
    condition_key="timepoint",
    group0="D8",
    group1="D21",
    basis="umap_scanorama",
    paga_key="paga",
    min_cells=20,
    label_top_n=8,
    ring_mode="none",
    fill_color_mode="delta_frac",
    arrow_color_mode="constant"
)